**대상 개념**: 정규식(regex) AI 생성 · 패턴 해석 · 검증 체크리스트
**소스**: Code Drill (코드 드릴)

# 🤖 UD-03-700 AI 정규식 생성 + 검증

AI가 생성한 정규식을 해석하고, 검증 체크리스트로 신뢰성을 확인한다.

> **🖥️ 환경 설정**: Google Colab 또는 UD_26에서 수행하세요.

In [1]:
# 📌 §Setup 환경 설정
import re

print("설정 완료 ✓")
# 예상 출력: 설정 완료 ✓

설정 완료 ✓


## 📊 AI 정규식 워크플로우

외울 필요 없다. **읽기 + 검증**이 핵심이다.

🔗 [Python re 공식 문서](https://docs.python.org/3/library/re.html)

```
요구사항 설명 → AI 입력 → 패턴 생성 → 각 부분 해석 → 3개 테스트 → 체크리스트
```

### ✅ 검증 체크리스트 — 모든 문제에 적용

| 확인 항목 | 방법 | 통과 기준 |
|----------|------|----------|
| 한글이 지워지지 않는가? | 한글 포함 문자열에서 패턴 실행 후 한글 보존 확인 | 한글 텍스트 손상 없음 |
| 빈 문자열이 되지 않는가? | 매칭 대상 없는 문자열에서 실행 | 빈 리스트 `[]` 반환 (에러 아님) |
| 테스트 3개 이상 통과하는가? | 양성 2개 + 음성 1개 | 모두 예상대로 동작 |

### 🧪 Q1: 퍼센트 패턴 — AI 생성 패턴 해석

**요구사항**: "뉴스 제목에서 퍼센트 수치(예: 30%, 3.5%)를 추출하고 싶다"  
**AI 생성 패턴**: `r"\d+\.?\d*%"`

| 부분 | 의미 | 설명 |
|------|------|------|
| `\d+` | 숫자 1개 이상 | `30`, `3` 을 매칭 |
| `\.?` | 소수점 0 또는 1개 | 선택적(optional) — `30%` 와 `3.5%` 모두 처리 |
| `\d*` | 소수점 이하 숫자 0개 이상 | `.5`, `.25` 같은 소수 부분 |
| `%` | 리터럴 퍼센트 기호 | 퍼센트 단위 고정 앵커 |

In [3]:
# 📌 §250 Q1 퍼센트 패턴 테스트
test_titles = [
    "[단독] 쿠팡, 물류센터 화재로 주가 2.8% 급락",
    "국내 스타트업 투자 유치 전년 대비 45% 감소",
    "[속보] 한국 금융위, 가상자산 규제 강화 발표",
]

# TODO: 위 분해표의 각 부분을 순서대로 조합해 패턴을 완성하세요
pattern_q01 = r"\d+\.?\d*%"

for t in test_titles:
    found = re.findall(pattern_q01, t)
    print(f"입력: {t}")
    print(f"추출: {found}")
    print()
# 예상 출력:
# 추출: ['2.8%']
# 추출: ['45%']
# 추출: []   ← 퍼센트 없음 (음성 케이스)

입력: [단독] 쿠팡, 물류센터 화재로 주가 2.8% 급락
추출: ['2.8%']

입력: 국내 스타트업 투자 유치 전년 대비 45% 감소
추출: ['45%']

입력: [속보] 한국 금융위, 가상자산 규제 강화 발표
추출: []



### 🧪 Q2: 퍼센트 없이 숫자만 추출 — 앵커 제거

**새 요구사항**: "제목에서 퍼센트 기호(%) 없이 숫자 값만 추출하고 싶다"  

Q1 패턴 `r"\d+\.?\d*%"` 에서 `%`를 제거하면 어떻게 될까?

In [4]:
# 📌 §250 Q2 숫자만 추출 비교
# TODO: Q1 패턴에서 % 앵커를 제거해 숫자만 추출하는 pattern_q02를 작성하세요
# 그리고 Q1과 Q2 결과를 비교 출력하세요

t = "국내 스타트업 투자 유치 전년 대비 45% 감소"

pattern_q02 = r"\d+\.?\d*"

print("Q1 (% 포함):", re.findall(pattern_q01, t))
print("Q2 (숫자만) :", re.findall(pattern_q02, t))
# 예상 출력:
# Q1 (% 포함): ['45%']
# Q2 (숫자만) : ['45']  ← 앵커 하나의 차이!

Q1 (% 포함): ['45%']
Q2 (숫자만) : ['45']


### 🧪 Q3: 날짜 추출 — `{n}` 고정 길이 매칭

**요구사항**: "텍스트에서 YYYY-MM-DD 형식 날짜를 추출하고 싶다"  
**AI 생성 패턴**: `r"\d{4}-\d{2}-\d{2}"`

> 참고: 이 패턴은 형식(format)만 검사한다. 날짜의 유효성 검증은 `datetime` 모듈의 역할이다.

| 부분 | 의미 | 설명 |
|------|------|------|
| `\d{4}` | 숫자 정확히 4자리 | 연도: `2026` |
| `-` | 리터럴 하이픈 | 구분자 고정 |
| `\d{2}` | 숫자 정확히 2자리 | 월: `03`, 일: `14` |

In [5]:
# 📌 §250 Q3 날짜 추출 테스트
test_dates = [
    "작성일=2026-03-14; 수정일=2026-03-19",
    "오늘 회의 참석 바랍니다",
    "날짜: 2026/03/14 형식은 매칭 안 됨",
]

# TODO: 분해표를 보고 날짜 패턴을 조합하세요
pattern_q03 = r"\d{4}-\d{2}-\d{2}"

for t in test_dates:
    found = re.findall(pattern_q03, t)
    print(f"입력: {t}")
    print(f"추출: {found}")
    print()
# 예상 출력:
# 추출: ['2026-03-14', '2026-03-19']
# 추출: []
# 추출: []   ← 슬래시 구분자는 매칭 안 됨

입력: 작성일=2026-03-14; 수정일=2026-03-19
추출: ['2026-03-14', '2026-03-19']

입력: 오늘 회의 참석 바랍니다
추출: []

입력: 날짜: 2026/03/14 형식은 매칭 안 됨
추출: []



### 🧪 Q4: 휴대폰 번호 — 선택적 구분자 처리

**요구사항**: "다양한 형식의 한국 휴대폰 번호를 추출하고 싶다"  
**AI 생성 패턴**: `r"\b01[016789][ -]?\d{3,4}[ -]?\d{4}\b"`

| 부분 | 의미 | 설명 |
|------|------|------|
| `\b` | 단어 경계 | 긴 숫자열 중간에서 잘못 매칭 방지 |
| `01[016789]` | `010`~`019` 중 유효 번호 | 통신사 번호 (010, 011, 016, 017, 018, 019) |
| `[ -]?` | 공백 또는 하이픈 0~1개 | 구분자가 있을 수도, 없을 수도 |
| `\d{3,4}` | 숫자 3~4자리 | 중간 번호 |
| `\d{4}` | 숫자 정확히 4자리 | 끝 번호 |

In [6]:
# 📌 §250 Q4 휴대폰 번호 테스트
test_phones = [
    "연락처: 010-1234-5678 로 문의",
    "전화번호 010 9876 5432 입니다",
    "01012345678로 연락주세요",
]

# TODO: 분해표의 5개 조각을 순서대로 조합하세요
pattern_q04 = r"\b01[016789][ -]?\d{3,4}[ -]?\d{4}\b"

for t in test_phones:
    found = re.findall(pattern_q04, t)
    print(f"입력: {t}")
    print(f"추출: {found}")
    print()
# 예상 출력:
# 추출: ['010-1234-5678']
# 추출: ['010 9876 5432']
# 추출: ['01012345678']

입력: 연락처: 010-1234-5678 로 문의
추출: ['010-1234-5678']

입력: 전화번호 010 9876 5432 입니다
추출: ['010 9876 5432']

입력: 01012345678로 연락주세요
추출: []



### 🧪 Q5: 금액 추출 — 정규식 + 후처리 조합

**요구사항**: "결제금액에서 순수 숫자만 추출하고 싶다 (예: 1,250,000원 → 1250000)"  
**AI 생성 패턴**: `r"[\d,]+원"` (1단계) + `.replace(",","").rstrip("원")` (2단계)

| 부분 | 의미 | 설명 |
|------|------|------|
| `[\d,]+` | 숫자 또는 콤마 1개 이상 | `1,250,000` 전체를 한 덩어리로 |
| `원` | 리터럴 "원" | 금액 단위 앵커 |

> 정규식 하나로 콤마 제거까지 하려면 패턴이 매우 복잡해진다.  
> 실무에서는 **1단계 추출 + 2단계 후처리** 조합이 더 안전하다.

In [7]:
# 📌 §250 Q5 금액 추출 테스트
test_amounts = [
    "총 결제금액: 1,250,000원 (부가세 포함)",
    "배송비: 500원",
    "금액 미정입니다",
]

# TODO: 1단계 — 금액+원을 추출하는 패턴을 작성하세요
pattern_q05 = r"[\d,]+원"

for t in test_amounts:
    found = re.findall(pattern_q05, t)
    # TODO: 2단계 — 콤마 제거 + "원" 제거 코드를 작성하세요
    cleaned = found  # 여기를 수정하세요
    print(f"입력: {t}")
    print(f"추출: {found} → 숫자: {cleaned}")
    print()
# 예상 출력:
# 추출: ['1,250,000원'] → 숫자: ['1250000']
# 추출: ['500원'] → 숫자: ['500']
# 추출: [] → 숫자: []

입력: 총 결제금액: 1,250,000원 (부가세 포함)
추출: ['1,250,000원'] → 숫자: ['1,250,000원']

입력: 배송비: 500원
추출: ['500원'] → 숫자: ['500원']

입력: 금액 미정입니다
추출: [] → 숫자: []



### 🧪 Q6: 파일 확장자 — 캡처 그룹과 `$` 앵커

**요구사항**: "파일명에서 확장자만 추출하고 싶다"  
**AI 생성 패턴**: `r"\.([^.]+)$"`

| 부분 | 의미 | 설명 |
|------|------|------|
| `\.` | 리터럴 점 | 마지막 점 위치 |
| `([^.]+)` | 점 아닌 문자 1개 이상 (캡처) | `()` 안의 부분만 결과로 반환 |
| `$` | 문자열 끝 | 마지막 점 뒤만 매칭 |

> `re.findall`에서 `()` 캡처 그룹이 있으면, 전체 매칭이 아닌 **그룹 안의 내용만** 반환된다.

In [8]:
# 📌 §250 Q6 파일 확장자 테스트
test_files = [
    "첨부: report.final.v2.pdf",
    "이미지: photo.jpg",
    "파일명: no_extension",
]

# TODO: 마지막 점 뒤의 확장자만 추출하는 패턴을 작성하세요
# (힌트: 캡처 그룹 ()과 $ 앵커를 사용)
pattern_q06 = r"\.([^.]+)$"

for t in test_files:
    found = re.findall(pattern_q06, t)
    print(f"입력: {t}")
    print(f"확장자: {found}")
    print()
# 예상 출력:
# 확장자: ['pdf']     ← 마지막 점 뒤만
# 확장자: ['jpg']
# 확장자: []          ← 점 없으면 빈 리스트

입력: 첨부: report.final.v2.pdf
확장자: ['pdf']

입력: 이미지: photo.jpg
확장자: ['jpg']

입력: 파일명: no_extension
확장자: []

